<a href="https://colab.research.google.com/github/swabhimansahu2004/Hybrid-TinyML-Environmental-Monitoring/blob/main/notebooks/Phase_6_Model_BenchMarking.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [6]:
import tensorflow as tf
import numpy as np
import pandas as pd
import os
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
import tensorflow as tf

# Now the rest of your code will work...

In [7]:
# 1. Load your original dataset
df = pd.read_csv('smoke_detection_iot.csv')

# 2. Features
features = ['Humidity[%]', 'Pressure[hPa]', 'Raw H2', 'NC2.5', 'NC1.0',
            'PM2.5', 'eCO2[ppm]', 'PM1.0', 'NC0.5', 'Temperature[C]',
            'TVOC[ppb]', 'Raw Ethanol']

X = df[features]
y = df['Fire Alarm']

# 3. Splits
X_train_full, X_test, y_train_full, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
X_train, X_val, y_train, y_val = train_test_split(X_train_full, y_train_full, test_size=0.125, random_state=42)

# 4. Scaling
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test) # <--- THIS WAS THE MISSING PIECE

print("✅ All data variables defined. You can now run the Benchmark cell.")

✅ All data variables defined. You can now run the Benchmark cell.


In [9]:
from sklearn.metrics import precision_score, recall_score, f1_score, accuracy_score
import time

def benchmark_model(model_path, X_test_data, y_test_data, is_tflite=False):
    # 1. Measure Memory
    size_kb = os.path.getsize(model_path) / 1024

    # 2. Performance Metrics Setup
    predictions = []
    iterations = min(500, len(X_test_data))
    start_time = time.time()

    if is_tflite:
        interpreter = tf.lite.Interpreter(model_path=model_path)
        interpreter.allocate_tensors()
        input_details = interpreter.get_input_details()
        output_details = interpreter.get_output_details()

        for i in range(iterations):
            sample = np.expand_dims(X_test_data[i], axis=0).astype(np.float32)
            interpreter.set_tensor(input_details[0]['index'], sample)
            interpreter.invoke()
            res = interpreter.get_tensor(output_details[0]['index'])
            # Convert probability to binary class (0 or 1)
            predictions.append(1 if res[0][0] > 0.5 else 0)
    else:
        model = tf.keras.models.load_model(model_path)
        for i in range(iterations):
            sample = np.expand_dims(X_test_data[i], axis=0)
            res = model.predict(sample, verbose=0)
            predictions.append(1 if res[0][0] > 0.5 else 0)

    end_time = time.time()
    avg_latency_ms = ((end_time - start_time) / iterations) * 1000

    # 3. Calculate Scientific Metrics
    y_true = y_test_data[:iterations]
    acc = accuracy_score(y_true, predictions)
    prec = precision_score(y_true, predictions)
    rec = recall_score(y_true, predictions)
    f1 = f1_score(y_true, predictions)

    return size_kb, avg_latency_ms, acc, prec, rec, f1

# --- RUN THE UPDATED BENCHMARK ---
# Assuming y_test contains your true labels (Fire/No Fire)
t_metrics = benchmark_model('teacher_model.h5', X_test_scaled, y_test)
s_metrics = benchmark_model('fire_model_optimized.tflite', X_test_scaled, y_test, is_tflite=True)

print(f"\n📊 FINAL COMPARATIVE ANALYSIS (Group 44-09):")
print(f"{'Metric':<20} | {'Teacher Model':<15} | {'Optimized Student':<15}")
print("-" * 60)
metrics_names = ['Size (KB)', 'Latency (ms)', 'Accuracy', 'Precision', 'Recall', 'F1-Score']
for i, name in enumerate(metrics_names):
    print(f"{name:<20} | {t_metrics[i]:<15.4f} | {s_metrics[i]:<15.4f}")

print(f"\n🚀 Optimization Summary: Speedup of {t_metrics[1]/s_metrics[1]:.1f}x with a {(t_metrics[0]-s_metrics[0])/t_metrics[0]*100:.1f}% size reduction.")


📊 FINAL COMPARATIVE ANALYSIS (Group 44-09):
Metric               | Teacher Model   | Optimized Student
------------------------------------------------------------
Size (KB)            | 271.2656        | 3.6875         
Latency (ms)         | 61.6400         | 0.0867         
Accuracy             | 0.9980          | 0.9720         
Precision            | 1.0000          | 0.9971         
Recall               | 0.9972          | 0.9641         
F1-Score             | 0.9986          | 0.9803         

🚀 Optimization Summary: Speedup of 710.7x with a 98.6% size reduction.


/usr/local/lib/python3.12/dist-packages/tensorflow/lite/python/interpreter.py:457: UserWarning:     Warning: tf.lite.Interpreter is deprecated and is scheduled for deletion in
    TF 2.20. Please use the LiteRT interpreter from the ai_edge_litert package.
    See the [migration guide](https://ai.google.dev/edge/litert/migration)
    for details.
    
  warnings.warn(_INTERPRETER_DELETION_WARNING)
